In [ ]:
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

print(text[:500])

In [ ]:
import torch

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}


def encode(s):
    return [stoi[c] for c in s]


def decode(l):
    return ''.join([itos[i] for i in l])


data = torch.tensor(encode(text), dtype=torch.long)

In [ ]:
seq_len = 50


def get_batch():
    i = torch.randint(len(data) - seq_len - 1, (1,))
    x = data[i:i + seq_len]
    y = data[i + 1:i + seq_len + 1]
    return x.unsqueeze(0), y.unsqueeze(0)

In [ ]:
import torch.nn as nn


class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):
        x = self.embed(x)
        out, h = self.rnn(x, h)
        out = self.fc(out)
        return out, h

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):
        x = self.embed(x)
        out, h = self.lstm(x, h)
        out = self.fc(out)
        return out, h

In [ ]:
def train(model, epochs=2000):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    criterion = nn.CrossEntropyLoss()

    losses = []

    for epoch in range(epochs):
        x, y = get_batch()

        logits, _ = model(x)

        loss = criterion(
            logits.view(-1, vocab_size),
            y.view(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 200 == 0:
            print(f"step {epoch:4}, loss {loss.item():.4f}")

        losses.append(loss.item())

    return losses

In [ ]:
def generate(model, start="To be", length=200, temperature=1):
    model.eval()

    x = torch.tensor([encode(start)], dtype=torch.long)
    h = None

    result = list(start)

    for _ in range(length):
        logits, h = model(x, h)

        # no temp
        # probs = torch.softmax(logits, dim=-1)

        # temp
        logits = logits[:, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)

        idx = torch.multinomial(probs, num_samples=1).item()

        result.append(itos[idx])

        x = torch.tensor([[idx]])

    return ''.join(result)

In [ ]:
# RNN
rnn_model = CharRNN(vocab_size, 128)

In [ ]:
rnn_losses  = train(rnn_model)

In [ ]:
print(generate(rnn_model))

In [ ]:
# LSTM
lstm_model = CharLSTM(vocab_size, 128)

In [ ]:
lstm_losses = train(lstm_model)

In [ ]:
print(generate(lstm_model))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(rnn_losses, label="RNN")
plt.plot(lstm_losses, label="LSTM")
plt.legend()
plt.title("Loss comparison")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.show()